# SARSA on CarRacing: A Realistic Example

## Learning Goals

In this notebook, you will see:

- How **SARSA** (from the Blackjack exercise) applies to a realistic Gymnasium environment
- How to **discretize continuous observations** (96×96 RGB images) into a simple state space
- The **same SARSA algorithm** working in a much more complex domain
- **Video visualization** of the learned policy in action

This demonstrates how the tabular RL concepts you learned extend to real environments!

## Environment: CarRacing-v2

We use Gymnasium's CarRacing environment with simplifications:

- **Original observation**: 96×96 RGB image (too complex for tabular methods!)
- **Our simplified state**: We extract just 3 key features:
  1. **Speed bin** (0-3): slow, medium, fast, very fast
  2. **Track angle** (-2 to +2): sharp left, left, straight, right, sharp right
  3. **On grass** (0 or 1): whether car is off the track
  
- **Actions** (discrete):
  - 0: do nothing
  - 1: steer left
  - 2: steer right
  - 3: gas (accelerate)
  - 4: brake

- **Rewards**: The environment gives small positive rewards for staying on track and making progress, negative for going off track.

- **Episode**: Ends after 1000 steps or if the car gets stuck off-track.

## Setup Instructions

**Recommended:** Use the conda environment provided in this folder.

If you haven't set it up yet:
```bash
conda env create -f environment.yml
conda activate sarsa-rl
jupyter notebook
```

The environment includes all necessary dependencies (gymnasium[box2d], imageio, imageio-ffmpeg).

In [1]:
# If not using the conda environment, you can install dependencies here:
# !pip install swig
# !pip install gymnasium[box2d] imageio imageio-ffmpeg

## Import Libraries

In [2]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='pygame.pkgdata')

import random
import gymnasium as gym
import numpy as np
from IPython.display import Video
import imageio

## State Discretization

This function converts the complex 96×96 RGB image into a simple discrete state.

It extracts just 3 key features:
- **Speed** (binned into 4 levels: slow, medium, fast, very fast)
- **Track direction** (binned into 5 angles: sharp left, left, straight, right, sharp right)
- **On grass** (binary: on track or off track)

This gives us a manageable state space of 4 × 5 × 2 = **40 states** total, making tabular SARSA feasible!

In [3]:
def discretize_observation(obs, env_state):
    """
    Convert 96x96 RGB image to a discrete state (speed_bin, angle_bin, on_grass).
    Returns a tuple that can be used as a state index.
    """
    # Extract car speed from environment (stored in env.car.hull.linearVelocity)
    # We bin it into 4 levels: 0=slow, 1=medium, 2=fast, 3=very fast
    speed = np.linalg.norm(env_state['speed']) if 'speed' in env_state else 0
    speed_bin = min(int(speed / 25), 3)
    
    # Detect track angle by looking at bottom center of image
    # Green pixels = on track, gray = road, white = outside
    bottom_strip = obs[80:90, 40:56, :]
    green_left = np.sum(bottom_strip[:, :8, 1] > 100)
    green_right = np.sum(bottom_strip[:, 8:, 1] > 100)
    
    if green_left > green_right + 20:
        angle_bin = 0  # sharp left
    elif green_left > green_right + 5:
        angle_bin = 1  # left
    elif green_right > green_left + 20:
        angle_bin = 4  # sharp right
    elif green_right > green_left + 5:
        angle_bin = 3  # right
    else:
        angle_bin = 2  # straight
    
    # Check if on grass (green pixels in bottom strip)
    green_pixels = np.sum(bottom_strip[:, :, 1] > 150)
    on_grass = 1 if green_pixels > 50 else 0
    
    return (speed_bin, angle_bin, on_grass)

def state_to_index(state):
    """
    Convert state tuple (speed_bin, angle_bin, on_grass) to a single integer index.
    This allows us to use a simple list for the Q-table.
    """
    speed_bin, angle_bin, on_grass = state
    return speed_bin * 10 + angle_bin * 2 + on_grass

NUM_STATES = 4 * 5 * 2  # 40 states total
NUM_ACTIONS = 5         # 5 discrete actions

## Environment Wrapper

This wraps the CarRacing environment to:
- Use **discrete actions** instead of continuous (5 actions: do nothing, left, right, gas, brake)
- Provide **simplified discrete states** via the discretization above
- Limit episode length to 1000 steps for faster training

In [4]:
class SimpleCarRacingWrapper:
    def __init__(self, render_mode=None):
        self.env = gym.make('CarRacing-v3', continuous=False, render_mode=render_mode)
        self.max_steps = 1000
        self.current_step = 0
        
    def reset(self):
        obs, info = self.env.reset()
        self.current_step = 0
        # Get initial state
        env_state = {'speed': [0, 0]}
        state = discretize_observation(obs, env_state)
        return state_to_index(state)
    
    def take_action(self, action):
        """
        Returns: (new_state_index, reward, game_over)
        """
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.current_step += 1
        
        # Get speed from environment
        try:
            speed = self.env.car.hull.linearVelocity
            env_state = {'speed': [speed.x, speed.y]}
        except:
            env_state = {'speed': [0, 0]}
        
        state = discretize_observation(obs, env_state)
        state_index = state_to_index(state)
        
        game_over = terminated or truncated or (self.current_step >= self.max_steps)
        
        return state_index, reward, game_over
    
    def close(self):
        self.env.close()

## Pre-Initialized Q-Table Helper

Just like in Blackjack, we pre-allocate the Q-table:

- **States**: 40 discrete states (speed × angle × on_grass)
- **Actions**: 5 discrete actions
- **Q**: A list of length 40, where `Q[s]` is a list of 5 Q-values

All Q-values start at 0.0.

In [5]:
Q_INIT_VALUE = 0.0

def make_q_table():
    Q = []
    for _ in range(NUM_STATES):
        Q.append([Q_INIT_VALUE] * NUM_ACTIONS)
    return Q

## SARSA Implementation

The same SARSA algorithm from Blackjack, adapted to CarRacing!

Key differences from Blackjack:
- **5 actions** instead of 2 (do nothing, left, right, gas, brake)
- **40 states** instead of 22 (speed × angle × on_grass)
- **gamma = 0.99** instead of 1.0 (episodes are longer, so we discount future rewards slightly)
- Uses `env.reset()` and `env.take_action()` to interact with the environment

In [6]:
def choose_action(Q, state, epsilon):
    """
    Epsilon-greedy over actions {0,1,2,3,4} using Q[state][action].
    - With probability epsilon: pick a random action (0-4).
    - Otherwise: pick the action with the largest Q-value.
    """
    if random.random() < epsilon:
        return random.randint(0, NUM_ACTIONS - 1)
    
    best_action = 0
    for a in range(NUM_ACTIONS):
        if Q[state][a] > Q[state][best_action]:
            best_action = a
    return best_action


def train_sarsa(episodes=100, alpha=0.1, gamma=0.99, epsilon=0.1):
    """
    Train a tabular SARSA agent on CarRacing and return Q.
    Same algorithm as Blackjack!
    """
    Q = make_q_table()
    env = SimpleCarRacingWrapper()

    for episode in range(episodes):
        s = env.reset()
        a = choose_action(Q, s, epsilon)
        
        total_reward = 0
        
        while True:
            s_next, r, game_over = env.take_action(a)
            total_reward += r
            
            # SARSA update (same as Blackjack!)
            if game_over:
                target = r
            else:
                a_next = choose_action(Q, s_next, epsilon)
                target = r + gamma * Q[s_next][a_next]
            
            Q[s][a] = Q[s][a] + alpha * (target - Q[s][a])

            if game_over:
                break
                
            s = s_next
            a = a_next
        
        if episode % 10 == 0:
            print(f"Episode {episode}, Total Reward: {total_reward:.2f}")
    
    env.close()
    return Q

## Train the Agent

Run this cell to train the SARSA agent on CarRacing.

**Note**: Training takes a few minutes (100 episodes × ~1000 steps each). You can reduce episodes for testing (e.g., `episodes=10`).

In [10]:
print("Training SARSA agent on CarRacing...")
Q_trained = train_sarsa(episodes=100, alpha=0.1, gamma=0.99, epsilon=0.1)
print("Training complete!")

Training SARSA agent on CarRacing...
Episode 0, Total Reward: -21.99
Episode 10, Total Reward: -36.75
Episode 20, Total Reward: -19.71
Episode 30, Total Reward: -30.23
Episode 40, Total Reward: -35.58
Episode 50, Total Reward: -58.33
Episode 60, Total Reward: -32.29
Episode 70, Total Reward: -24.73
Episode 80, Total Reward: -52.21
Episode 90, Total Reward: -2.82
Training complete!


## Visualize the Learned Policy

Let's see how well the agent learned! This will:
- Run the agent for 1 episode using the greedy policy (epsilon=0)
- Record a video of the episode
- Display the video in the notebook

In [9]:
def visualize_policy(Q, filename='carracing_sarsa.mp4'):
    """
    Run one episode with the greedy policy and save as video.
    """
    env = SimpleCarRacingWrapper(render_mode='rgb_array')
    frames = []
    
    s = env.reset()
    game_over = False
    total_reward = 0
    
    while not game_over:
        # Greedy action (epsilon = 0)
        a = choose_action(Q, s, epsilon=0.0)
        
        # Capture frame
        frame = env.env.render()
        frames.append(frame)
        
        # Take action
        s_next, r, game_over = env.take_action(a)
        total_reward += r
        s = s_next
    
    env.close()
    
    # Save video
    imageio.mimsave(filename, frames, fps=30)
    print(f"Video saved to {filename}")
    print(f"Total reward: {total_reward:.2f}")
    
    return filename

# Run visualization
try:
    video_file = visualize_policy(Q_trained)
    Video(video_file, embed=True, width=400)
except Exception as e:
    print("Visualization error:", e)

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Video saved to carracing_sarsa.mp4
Total reward: -9.09


## Discussion Questions

After running the demo, think about:

1. **How does the car's behavior change as training progresses?**  
   Watch the rewards increase over episodes. What strategies does the agent learn?

2. **Why is `gamma=0.99` instead of `gamma=1.0` (like in Blackjack)?**  
   CarRacing episodes can be very long (1000 steps). What does discounting mean here?

3. **How good is our state discretization?**  
   We reduced a 96×96×3 = 27,648-dimensional image to just 40 discrete states! What information are we losing? Could we do better?

4. **Compare to Blackjack:**  
   - What's similar about the SARSA algorithm?
   - What's different about the environment?
   - Why is CarRacing much harder to solve?

5. **Limitations of tabular SARSA:**  
   Why can't we use 27,648 states (one per pixel combination)? What would we need instead? (Hint: function approximation, deep RL)